# *MOV to MP4 Video Converter with Audio Muting Feature*

# *Installing Dependencies*

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!pip install -q tqdm

# *Mounting Google Drive (Optional)*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# *Import Libraries*

In [ ]:
import subprocess
import shlex
from pathlib import Path
import sys
from tqdm import tqdm

# *Run Shell Command*

In [ ]:
def run(cmd):
    proc = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    return proc.returncode, proc.stdout

# *GPU Accleration*

In [ ]:
def has_nvidia_gpu():
    rc, out = run("nvidia-smi -L")
    return rc == 0 and out.strip() != ""

# *Encoding*

In [ ]:
def ffmpeg_supports_nvenc():
    rc, out = run("ffmpeg -encoders")
    if rc != 0:
        return False
    return "h264_nvenc" in out or "hevc_nvenc" in out

# *Building*

In [ ]:
def build_command(in_path: Path, out_path: Path, use_nvenc: bool):
    """Return an ffmpeg command string to mute and convert to mp4."""
    inp = shlex.quote(str(in_path))
    out = shlex.quote(str(out_path))
    if use_nvenc:
        return f"ffmpeg -y -hwaccel cuda -i {inp} -an -c:v h264_nvenc -preset p7 {out}"
    else:
        return f"ffmpeg -y -i {inp} -an -c:v libx264 -preset fast -crf 23 {out}"

# *Folder Conversion and Finding .MOV Files*

In [ ]:
def convert_folder(input_folder, output_folder, recursive=True):
    input_folder = Path(input_folder).expanduser().resolve()
    output_folder = Path(output_folder).expanduser().resolve()
    if not input_folder.exists():
        raise FileNotFoundError(f"Input folder not found: {input_folder}")
    output_folder.mkdir(parents=True, exist_ok=True)

    gpu = has_nvidia_gpu()
    nvenc = ffmpeg_supports_nvenc()
    use_nvenc = gpu and nvenc

    print(f"GPU detected: {gpu}")
    print(f"ffmpeg NVENC support: {nvenc}")
    print(f"Using NVENC: {use_nvenc}\n")

    pattern = "**/*.MOV" if recursive else "*.MOV"
    files_upper = list(input_folder.glob(pattern))
    pattern_low = "**/*.mov" if recursive else "*.mov"
    files_lower = list(input_folder.glob(pattern_low))
    files = sorted({p: None for p in (files_upper + files_lower)}.keys())

    if not files:
        print("No .MOV files found. Nothing to convert.")
        return

    for src in tqdm(files, desc="Converting files"):
        rel = src.relative_to(input_folder)
        dst = (output_folder / rel).with_suffix(".mp4")
        dst.parent.mkdir(parents=True, exist_ok=True)

        cmd = build_command(src, dst, use_nvenc)
        rc, out = run(cmd)
        if rc != 0:
            print(f"\nFAILED: {src}\nffmpeg output:\n{out}\n")
        else:
            print(f"Saved: {dst}")

# *Main*

In [ ]:
if __name__ == "__main__":
    input_folder = input("Enter input folder path: ").strip()
    output_folder = input("Enter output folder path: ").strip()
    convert_folder(input_folder, output_folder, recursive=True)